In [1]:
#conda activate durham_ml_venv

import rasterio
from rasterio.features import rasterize
import numpy as np
import osmnx as ox
import geopandas as gpd
from scipy.ndimage import distance_transform_edt


In [3]:
# Input raster - use ndvi 10m? or svf
raster_path = "NDVI_2024SummerMosaic_final.tif"
output_path = "distance_to_main_roads.tif"

# 1. Load raster
with rasterio.open(raster_path) as src:
    ndvi = src.read(1, masked=True)   # read NDVI band
    ndvi_meta = src.meta.copy()
    transform = src.transform
    crs = src.crs
    out_shape = ndvi.shape

# 2. Download/filter roads
cf = '["highway"~"motorway|motorway_link|trunk|trunk_link|primary|primary_link"]'
G = ox.graph_from_place("Durham County, North Carolina, USA", 
                        network_type='drive', simplify=True, 
                        retain_all=True, custom_filter=cf)

nodes, edges = ox.graph_to_gdfs(G)

# 3. Rasterize roads to binary mask
road_mask = rasterize(
    [(geom, 1) for geom in edges.geometry],
    out_shape=out_shape,
    transform=transform,
    fill=0,
    dtype="uint8"
)

# 4. Compute distance transform (pixel units → meters via transform scale)
# Distance to nearest road pixel (0 = on a road)
dist_pixels = distance_transform_edt(road_mask == 0)
pixel_size = transform.a  # assuming square pixels
dist_meters = dist_pixels * pixel_size

# Mask out nodata regions (where NDVI is nodata)
#dist_meters = np.where(ndvi.mask, np.nan, dist_meters)

# 5. Save result as GeoTIFF
ndvi_meta.update({
    "dtype": "float32",
    "count": 1,
    "nodata": np.nan
})

with rasterio.open(output_path, "w", **ndvi_meta) as dst:
    dst.write(dist_meters.astype("float32"), 1)

print(f"Saved distance raster: {output_path}")


Saved distance raster: distance_to_main_roads.tif


In [ ]:
## Loop through and create the distance to roads for the other cities 

#Philadelphia, Chicago, Phoenix, Baltimore 

for city_name in ['Philadelphia', 'Chicago', 'Phoenix', 'Baltimore']:

    # Input raster - use ndvi 10m? or svf
    raster_path = f"{city_name}_NDVI_2024SummerMosaic.tif"
    output_path = f"{city_name}_distance_to_main_roads.tif"

    # 1. Load raster
    with rasterio.open(raster_path) as src:
        ndvi = src.read(1, masked=True)   # read NDVI band
        ndvi_meta = src.meta.copy()
        transform = src.transform
        crs = src.crs
        out_shape = ndvi.shape

    # 2. Download/filter roads
    cf = '["highway"~"motorway|motorway_link|trunk|trunk_link|primary|primary_link"]'
    G = ox.graph_from_place("Durham County, North Carolina, USA", 
                        network_type='drive', simplify=True, 
                        retain_all=True, custom_filter=cf)

    nodes, edges = ox.graph_to_gdfs(G)

    # 3. Rasterize roads to binary mask
    road_mask = rasterize(
        [(geom, 1) for geom in edges.geometry],
        out_shape=out_shape,
        transform=transform,
        fill=0,
        dtype="uint8"
    )

    # 4. Compute distance transform (pixel units → meters via transform scale)
    # Distance to nearest road pixel (0 = on a road)
    dist_pixels = distance_transform_edt(road_mask == 0)
    pixel_size = transform.a  # assuming square pixels
    dist_meters = dist_pixels * pixel_size

    # Mask out nodata regions (where NDVI is nodata)
    #dist_meters = np.where(ndvi.mask, np.nan, dist_meters)

    # 5. Save result as GeoTIFF
    ndvi_meta.update({
        "dtype": "float32",
        "count": 1,
        "nodata": np.nan
    })

    with rasterio.open(output_path, "w", **ndvi_meta) as dst:
        dst.write(dist_meters.astype("float32"), 1)

    print(f"Saved distance raster: {output_path}")


In [ ]:
######### Below for the svf based, higher res ##########

In [2]:
import os
import rioxarray as rxr
import osmnx as ox
import geopandas as gpd
from shapely.geometry import box

# paths
svf_fp = "svf_merged.tif"
roads_out = "roads_clipped_simplified.geojson"

# 1) read svf mosaic extent / crs
svf = rxr.open_rasterio(svf_fp, masked=True)
crs = svf.rio.crs
bounds = svf.rio.bounds()   # (left, bottom, right, top)

# 2) fetch road network with OSMnx for the area (you can use the county or bbox)
# Use the place query if that works for your area, or use bbox via ox.graph_from_bbox.
# Here we use a bounding box to limit to the same area as the svf
north = bounds[3]
south = bounds[1]
east  = bounds[2]
west  = bounds[0]

# OSMnx expects lat/long (WGS84). If your SVF is EPSG:32617, transform bbox to WGS84
bbox_gdf = gpd.GeoDataFrame({"geometry":[box(west, south, east, north)]}, crs=crs)
bbox_wgs84 = bbox_gdf.to_crs("EPSG:4326")
minx, miny, maxx, maxy = bbox_wgs84.total_bounds  # lon_min, lat_min, lon_max, lat_max

# 3) get graph filtered to main roads
cf = '["highway"~"motorway|motorway_link|trunk|trunk_link|primary|primary_link"]'
bbox = (maxy, miny, maxx, minx)

G = ox.graph_from_bbox(
    bbox,
    network_type="drive",
    simplify=False,
    retain_all=True,
    custom_filter=cf
)
nodes, edges = ox.graph_to_gdfs(G)

# 4) reproject edges to the SVF CRS (so GDAL rasterize uses same CRS)
edges = edges.to_crs(crs)

# 5) optional: simplify geometries to reduce complexity (tolerance in map units; e.g., 5 meters)
edges["geometry"] = edges["geometry"].simplify(tolerance=5.0, preserve_topology=True)

# 6) write GeoJSON (small, clipped, simplified)
edges = edges[~edges.geometry.is_empty]
edges.to_file(roads_out, driver="GeoJSON")
print("Saved", roads_out)


/nas/longleaf/home/sberk/.conda/envs/durham_ml_venv/lib/python3.13/site-packages/osmnx/_overpass.py:271: UserWarning: This area is 38,838 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


ConnectTimeout: HTTPSConnectionPool(host='overpass-api.de', port=443): Max retries exceeded with url: /api/interpreter (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7ffa4bcf5950>, 'Connection to overpass-api.de timed out. (connect timeout=180)'))